# Reset: back to a clean workspace

Drops the tables so the episodes can be recorded again **from the beginning**.

| Flag | What gets deleted | What stays | Start again from |
|---|---|---|---|
| `DROP_TABLES` | every table and view in `<project>_<env>` + the contents of its volumes (Auto Loader checkpoints) | catalogs, schemas, volumes, landing files | `02_bronze_tables` |
| `CLEAR_LANDING` | every file in the landing volumes | the volumes themselves | `00_kickoff`, step 7 (backfill) |

Catalogs, schemas and volumes are **never** dropped, so there is no need to run the Unity Catalog bootstrap again.

**Nothing is deleted until you set `CONFIRM = True`.** Run the notebook once without it and read the list of what would go.

In [0]:
CATALOG_PREFIX = "moustache_markt"
ENV = "dev"

DROP_TABLES = True      # tables + checkpoints in the environment catalog
CLEAR_LANDING = False   # source files - regenerating them takes a few minutes
CONFIRM = False         # False = only show what would be deleted

SOURCES = f"{CATALOG_PREFIX}_sources"
TARGET = f"{CATALOG_PREFIX}_{ENV}"

# A reset for recordings should never touch production by accident
assert ENV != "prod", "This notebook does not reset prod."

## What would be deleted

In [0]:
tables = spark.sql(
    """
    SELECT table_schema, table_name, table_type
    FROM system.information_schema.tables
    WHERE table_catalog = :catalog AND table_schema <> 'information_schema'
    ORDER BY table_schema, table_name
    """,
    args={"catalog": TARGET},
).collect()

volumes = spark.sql(
    """
    SELECT volume_catalog, volume_schema, volume_name
    FROM system.information_schema.volumes
    WHERE volume_catalog IN (:target, :sources)
    ORDER BY volume_catalog, volume_schema, volume_name
    """,
    args={"target": TARGET, "sources": SOURCES},
).collect()

# Volumes in the environment catalog (checkpoints) go together with the tables,
# landing volumes only when CLEAR_LANDING is on.
volumes_to_clear = [
    v for v in volumes
    if (DROP_TABLES and v.volume_catalog == TARGET) or (CLEAR_LANDING and v.volume_catalog == SOURCES)
]

if DROP_TABLES:
    print(f"Tables and views in {TARGET}: {len(tables)}")
    for t in tables:
        print(f"  {t.table_type:<10} {t.table_schema}.{t.table_name}")

print(f"\nVolumes to empty: {len(volumes_to_clear)}")
for v in volumes_to_clear:
    path = f"/Volumes/{v.volume_catalog}/{v.volume_schema}/{v.volume_name}"
    print(f"  {path}  ({len(dbutils.fs.ls(path))} entries)")

## Delete

`DROP TABLE` on a managed table removes the data as well, not only the definition.
Unity Catalog keeps it for 7 days, so a table dropped by mistake can still be brought back with `UNDROP TABLE <name>`.

Files removed from a volume with `dbutils.fs.rm` are gone for good.

In [0]:
if not CONFIRM:
    print("CONFIRM = False - nothing was deleted. Set CONFIRM = True in the config cell and run again.")
else:
    if DROP_TABLES:
        for t in tables:
            kind = "VIEW" if t.table_type == "VIEW" else "TABLE"
            spark.sql(f"DROP {kind} IF EXISTS {TARGET}.{t.table_schema}.{t.table_name}")
            print(f"dropped {t.table_schema}.{t.table_name}")

    for v in volumes_to_clear:
        path = f"/Volumes/{v.volume_catalog}/{v.volume_schema}/{v.volume_name}"
        # Empty the volume, but keep the volume itself
        for entry in dbutils.fs.ls(path):
            dbutils.fs.rm(entry.path, recurse=True)
        print(f"emptied {path}")

## Check

In [0]:
display(spark.sql(
    """
    SELECT table_schema, count(*) AS tables
    FROM system.information_schema.tables
    WHERE table_catalog = :catalog AND table_schema <> 'information_schema'
    GROUP BY table_schema
    """,
    args={"catalog": TARGET},
))   # empty result = no tables left